# S&P 500 Stock Analyzer (Yahoo Finance)

This notebook provides:

1) **Plot a stock's price chart** (Yahoo Finance)
2) **Retrieve P/E (trailing), Price/EBITDA, and Price/Book**
3) **S&P 500 P/E Ranking** — ordered list from lowest to highest P/E, with CSV export

> Data is pulled via `yfinance`. Some tickers may have missing fields; the code includes sensible fallbacks.


⚠️ **Important:** Always run the notebook **from the top, in order**. This ensures helper functions (`fetch_price_history`, `fetch_fundamentals`, etc.) are defined before you call them.

## 0) Setup (Run once if needed)

In [ ]:
# If packages are missing, run this cell, then restart the kernel and run all cells top-to-bottom.
# Remove the leading '!' if your environment uses %pip magic.
!pip install --quiet yfinance pandas matplotlib lxml tqdm

## 1) Imports & Utility Functions

## 1.a) Offline Mode (No Internet Required)

If your environment blocks the web, this notebook can still work using an **embedded S&P 500 ticker list**.
- You can edit the list below to be the complete set (copy-paste tickers separated by commas or one per line).
- The function will prefer cached `sp500_tickers_offline.json` if present; otherwise it uses the embedded list.
- If you later run online, the fetched list will be cached for future offline runs automatically.


In [ ]:
# <<< Edit EMBEDDED_SP500_TICKERS to the full 500 if you want a 100% offline run >>>
EMBEDDED_SP500_TICKERS = [
    # A compact starter set; extend this list to the full 500 (e.g., paste from a CSV)
    "AAPL","MSFT","AMZN","NVDA","GOOGL","GOOG","META","TSLA","BRK-B","UNH",
    "JNJ","XOM","JPM","PG","V","HD","LLY","AVGO","CVX","PEP","ABBV","KO","PFE",
    "MRK","BAC","MA","COST","TMO","MCD","CSCO","WMT","ABT","ADBE","CRM","ACN",
    "INTC","CMCSA","VZ","NFLX","WFC","TXN","UPS","LIN","AMGN","NEE","ORCL",
    "PM","IBM","HON","QCOM"
]

# Cache helpers
import json, os
CACHE_PATH = 'sp500_tickers_offline.json'

def _load_cached_tickers():
    if os.path.exists(CACHE_PATH):
        try:
            with open(CACHE_PATH, 'r') as f:
                data = json.load(f)
            if isinstance(data, list) and len(data) >= 450:
                print(f"[S&P 500] Loaded {len(data)} cached tickers from {CACHE_PATH}.")
                return data
        except Exception:
            pass
    return None

def _save_cached_tickers(tickers):
    try:
        with open(CACHE_PATH, 'w') as f:
            json.dump(tickers, f)
        print(f"[S&P 500] Cached {len(tickers)} tickers to {CACHE_PATH}.")
    except Exception:
        pass


In [ ]:

def get_sp500_tickers():
    """Robust S&P 500 retrieval with offline cache and embedded fallback."""
    import pandas as pd
    import yfinance as yf

    # 1) Try cache first
    cached = _load_cached_tickers()
    if cached:
        return cached

    # Helper
    def _normalize(ts):
        return [t.replace('.', '-').strip().upper() for t in ts if isinstance(t, str) and t.strip()]

    # 2) Try Slickcharts
    try:
        tables = pd.read_html('https://www.slickcharts.com/sp500')
        if tables and not tables[0].empty:
            col_candidates = [c for c in tables[0].columns if 'Symbol' in str(c)]
            if col_candidates:
                tickers = _normalize(tables[0][col_candidates[0]].tolist())
                if len(tickers) >= 450:
                    _save_cached_tickers(tickers)
                    print(f"[S&P 500] Loaded {len(tickers)} from Slickcharts.")
                    return tickers
    except Exception as e:
        print('[S&P 500] Slickcharts fetch failed:', e)

    # 3) Try Wikipedia
    try:
        tables = pd.read_html('https://en.wikipedia.org/wiki/List_of_S%26P_500_companies', flavor='lxml')
        if tables and not tables[0].empty:
            tickers = _normalize(tables[0]['Symbol'].tolist())
            if len(tickers) >= 450:
                _save_cached_tickers(tickers)
                print(f"[S&P 500] Loaded {len(tickers)} from Wikipedia.")
                return tickers
    except Exception as e:
        print('[S&P 500] Wikipedia fetch failed:', e)

    # 4) Try yfinance helper
    try:
        if hasattr(yf, 'tickers_sp500'):
            tickers = _normalize(list(yf.tickers_sp500()))
            if len(tickers) >= 450:
                _save_cached_tickers(tickers)
                print(f"[S&P 500] Loaded {len(tickers)} via yfinance helper.")
                return tickers
    except Exception as e:
        print('[S&P 500] yfinance helper failed:', e)

    # 5) Embedded fallback
    if len(EMBEDDED_SP500_TICKERS) >= 450:
        tickers = _normalize(EMBEDDED_SP500_TICKERS)
        print(f"[S&P 500] Using embedded list with {len(tickers)} tickers.")
        return tickers

    # If embedded is partial, still return it with a warning so the notebook works
    tickers = _normalize(EMBEDDED_SP500_TICKERS)
    print(f"[S&P 500] Using embedded partial list ({len(tickers)}). "
          "For a 100% offline run, paste the full 500 into EMBEDDED_SP500_TICKERS above ")
    return tickers


## 2) Single Ticker (Chart + Ratios)

In [ ]:

# --- Bootstrap: make this cell self-contained if you skipped earlier cells ---
try:
    _ = fetch_price_history  # type: ignore
    _ = fetch_fundamentals   # type: ignore
except NameError:
    import math, time, pandas as pd, matplotlib.pyplot as plt, yfinance as yf
    try:
        from tqdm import tqdm  # noqa: F401
    except Exception:
        pass

    def _safe_get(dct, *keys, default=None):
        cur = dct
        for k in keys:
            if cur is None:
                return default
            try:
                cur = cur.get(k)
            except AttributeError:
                return default
        return default if cur is None else cur

    def human_readable_number(x):
        if x is None or (isinstance(x, float) and math.isnan(x)):
            return 'N/A'
        for unit in ['', 'K', 'M', 'B', 'T']:
            if abs(x) < 1000.0:
                return f"{x:,.2f}{unit}"
            x /= 1000.0
        return f"{x:,.2f}P"

    def fetch_price_history(ticker: str, period='1y', interval='1d'):
        t = yf.Ticker(ticker)
        hist = t.history(period=period, interval=interval, auto_adjust=False)
        if hist is None or hist.empty:
            raise ValueError(f"No price history returned for {ticker}.")
        return hist

    def plot_price_chart(hist, ticker: str, show_volume: bool = True):
        close = hist['Close']
        ax = close.plot(title=f"{ticker} - Close Price", figsize=(10, 5))
        ax.set_xlabel('Date'); ax.set_ylabel('Price')
        plt.tight_layout(); plt.show()
        if show_volume and 'Volume' in hist.columns:
            ax2 = hist['Volume'].plot(title=f"{ticker} - Volume", figsize=(10, 3))
            ax2.set_xlabel('Date'); ax2.set_ylabel('Volume')
            plt.tight_layout(); plt.show()

    def fetch_fundamentals(ticker: str):
        t = yf.Ticker(ticker)
        try:
            info = t.get_info()
        except Exception:
            info = getattr(t, 'info', {}) or {}
        try:
            income_ttm = t.income_stmt(trailing=True)
        except Exception:
            income_ttm = None
        try:
            bs_annual = t.balance_sheet()
        except Exception:
            bs_annual = None

        price = _safe_get(info, 'currentPrice')
        shares_out = _safe_get(info, 'sharesOutstanding')
        if price is None:
            try:
                last = t.history(period='5d', interval='1d')
                if not last.empty:
                    price = float(last['Close'].iloc[-1])
            except Exception:
                pass
        market_cap = _safe_get(info, 'marketCap')
        if market_cap is None and price is not None and shares_out:
            market_cap = price * shares_out

        eps_ttm = _safe_get(info, 'trailingEps')
        pe_trailing = _safe_get(info, 'trailingPE')
        if pe_trailing is None and price not in (None, 0) and eps_ttm not in (None, 0):
            pe_trailing = price / eps_ttm

        bvps = _safe_get(info, 'bookValue')
        pb = _safe_get(info, 'priceToBook')
        if pb is None and price not in (None, 0) and bvps not in (None, 0):
            pb = price / bvps

        total_equity = None
        if bs_annual is not None and not bs_annual.empty:
            for candidate in ['Total Stockholder Equity', 'Total Equity Gross Minority Interest']:
                if candidate in bs_annual.index:
                    try:
                        total_equity = float(bs_annual.loc[candidate].iloc[0]); break
                    except Exception:
                        pass
        if pb is None and market_cap not in (None, 0) and total_equity not in (None, 0):
            pb = market_cap / total_equity

        ebitda_ttm = _safe_get(info, 'ebitda')
        if ebitda_ttm is None and income_ttm is not None and not income_ttm.empty:
            if 'EBITDA' in income_ttm.index:
                try: ebitda_ttm = float(income_ttm.loc['EBITDA'].iloc[0])
                except Exception: pass

        p_to_ebitda = None
        if market_cap not in (None, 0) and ebitda_ttm not in (None, 0):
            p_to_ebitda = market_cap / ebitda_ttm

        currency = _safe_get(info, 'currency') or _safe_get(info, 'financialCurrency')

        return {
            'ticker': ticker, 'price': price, 'market_cap': market_cap,
            'eps_ttm': eps_ttm, 'pe_trailing': pe_trailing,
            'bvps': bvps, 'price_to_book': pb, 'total_equity': total_equity,
            'ebitda_ttm': ebitda_ttm, 'price_to_ebitda': p_to_ebitda, 'currency': currency
        }

# --- Inputs ---
TICKER = 'AAPL'     # Example: AAPL, MSFT, TSLA
PERIOD = '1y'       # 1mo, 3mo, 6mo, 1y, 2y, 5y, max
INTERVAL = '1d'     # 1d, 1wk, 1mo
SHOW_VOLUME = True

hist = fetch_price_history(TICKER, period=PERIOD, interval=INTERVAL)
plot_price_chart(hist, TICKER, show_volume=SHOW_VOLUME)

facts = fetch_fundamentals(TICKER)
print(f"Ticker: {facts['ticker']}  |  Currency: {facts['currency']}")
print('-- Key Metrics --')
print('Price (last):', human_readable_number(facts['price']))
print('P/E (Trailing):', f"{facts['pe_trailing']:.2f}" if isinstance(facts['pe_trailing'], (int, float)) else 'N/A')
print('Price/EBITDA (MC/EBITDA):', f"{facts['price_to_ebitda']:.2f}" if isinstance(facts['price_to_ebitda'], (int, float)) else 'N/A')
print('Price/Book:', f"{facts['price_to_book']:.2f}" if isinstance(facts['price_to_book'], (int, float)) else 'N/A')


## 2.a) MACD (6, 12, 20 **weeks**) for Selected Ticker
    
This computes a weekly **MACD** using:
- **Fast EMA = 6 weeks**
- **Slow EMA = 12 weeks**
- **Signal EMA = 20 weeks**

It then plots **MACD**, **Signal**, and the **Histogram** (MACD − Signal) together for the ticker chosen in section 2.


In [ ]:

# --- Bootstrap: ensure we have price data & matplotlib ---
import pandas as pd, numpy as np, matplotlib.pyplot as plt

# Use the history already downloaded if available; otherwise fetch daily data
try:
    _ = hist  # from section 2
except NameError:
    try:
        hist = fetch_price_history(TICKER, period=PERIOD, interval='1d')
    except Exception:
        import yfinance as yf
        hist = yf.Ticker(TICKER).history(period='2y', interval='1d')

def weekly_macd(df_close: pd.Series, fast_weeks, slow_weeks, signal_weeks):
    """Compute weekly MACD using weekly-resampled closing prices."""
    wk = df_close.resample('W-FRI').last().dropna()
    ema_fast = wk.ewm(span=fast_weeks, adjust=False).mean()
    ema_slow = wk.ewm(span=slow_weeks, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal_weeks, adjust=False).mean()
    hist = macd_line - signal_line
    return pd.DataFrame({'Close (W)': wk, 'MACD': macd_line, 'Signal': signal_line, 'Histogram': hist}).dropna()

# Compute three MACDs with different fast/slow/signal = (6,12,20), (12,24,20), (20,40,20)
macd6 = weekly_macd(hist['Close'], fast_weeks=6, slow_weeks=12, signal_weeks=20)
macd12 = weekly_macd(hist['Close'], fast_weeks=12, slow_weeks=24, signal_weeks=20)
macd20 = weekly_macd(hist['Close'], fast_weeks=20, slow_weeks=40, signal_weeks=20)

# Align indexes
df_all = pd.DataFrame({
    'MACD(6w)': macd6['MACD'],
    'MACD(12w)': macd12['MACD'],
    'MACD(20w)': macd20['MACD'],
}).dropna()

# Plot all three MACD lines with distinct colors + legend
ax = df_all.plot(figsize=(10,5), color=['blue', 'orange', 'green'], lw=2,
                 title=f"{TICKER} - Weekly MACD Lines (6, 12, 20 weeks)")
ax.set_xlabel("Date"); ax.set_ylabel("MACD")
plt.legend(title="MACD Variants")
plt.tight_layout(); plt.show()

df_all.tail()


## 3) Retrieve S&P 500 List

In [ ]:

sp500_tickers = get_sp500_tickers()
print('Total S&P 500 tickers retrieved:', len(sp500_tickers))
sp500_tickers[:25]


## 4) Fundamentals for S&P 500 & P/E Ranking (Lowest → Highest)

In [ ]:

# --- P/E Ranking with Bootstrap + Logging ---
import math, time, pandas as pd

# Ensure tqdm exists
try:
    from tqdm import tqdm
except Exception:
    try:
        import sys, subprocess; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tqdm"])
        from tqdm import tqdm
    except Exception:
        # fallback to dummy tqdm
        def tqdm(x, **kwargs): return x

# Ensure fetch_fundamentals exists (bootstrap if missing)
try:
    _ = fetch_fundamentals  # noqa
except NameError:
    import yfinance as yf

    def _safe_get(dct, *keys, default=None):
        cur = dct
        for k in keys:
            if cur is None: return default
            try: cur = cur.get(k)
            except AttributeError: return default
        return default if cur is None else cur

    def fetch_fundamentals(ticker: str):
        t = yf.Ticker(ticker)
        try: info = t.get_info()
        except Exception: info = getattr(t, "info", {}) or {}
        price = _safe_get(info, "currentPrice")
        eps_ttm = _safe_get(info, "trailingEps")
        pe_trailing = _safe_get(info, "trailingPE")
        if pe_trailing is None and price not in (None,0) and eps_ttm not in (None,0):
            pe_trailing = price/eps_ttm
        return {"ticker": ticker, "price": price, "pe_trailing": pe_trailing,
                "market_cap": _safe_get(info, "marketCap"),
                "currency": _safe_get(info, "currency")}

# Main loop with logging
results, failed = [], []
for tk in tqdm(sp500_tickers, desc="Downloading fundamentals"):
    try:
        facts = fetch_fundamentals(tk)
        pe = facts["pe_trailing"]
        if isinstance(pe, (int, float)) and pe and pe > 0 and not math.isnan(pe):
            results.append(facts)
        else:
            failed.append((tk, "No valid P/E"))
    except Exception as e:
        failed.append((tk, str(e)))
    time.sleep(0.05)  # gentle throttle

pe_df = pd.DataFrame(results)
pe_df = pe_df.sort_values(by="pe_trailing", ascending=True).reset_index(drop=True)
print("Total with valid (positive) P/E:", len(pe_df))
print("Failed tickers:", len(failed))
if failed: print(failed[:10], "...")

pe_df.head(25)


### 4.1) Save Ordered P/E Table to CSV

In [ ]:
output_path = 'sp500_pe_ranking.csv'
pe_df.to_csv(output_path, index=False)
print('Saved:', output_path)

## 5) Notes

- **Price/EBITDA** here is defined as *Market Cap / EBITDA (TTM)*. Many analysts prefer **EV/EBITDA**.
- P/E computed from Yahoo fields (or price / EPS fallback). Negative or missing P/E is excluded from the ranking.
- **BRK.B → BRK-B** and similar dot tickers are converted automatically for Yahoo Finance.
- Always double-check critical numbers against official filings.
